<a href="https://colab.research.google.com/github/yu-hidaka/AD-DIFFI/blob/develop/examples/Real_World_Analysis/Real_World_Analysis_Hepatitis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Real-World Analysis: Hepatitis (Table S6)**
**Project: Adjust DIFFI (AD-DIFFI): Robust Feature Importance for Mixed-Type Data**


## Cell 1: Environment Setup

In [ ]:
# 1. Install required libraries
!pip install -q lifelines scikit-learn pandas numpy matplotlib tabulate

import os
import sys
import importlib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# --- 2. Repository Setup ---
REPO_NAME = "AD-DIFFI"
REPO_URL = "https://github.com/yu-hidaka/AD-DIFFI.git"
BRANCH = "develop"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_NAME}...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print(f"{REPO_NAME} already exists.")

# --- 3. Path & Structure Configuration ---
project_root = os.path.abspath(REPO_NAME)
project_src = os.path.join(project_root, 'src')

conflict_file = os.path.join(project_root, 'ad_diffi.py')
if os.path.exists(conflict_file):
    print(f"Renaming {conflict_file} to avoid import conflict.")
    os.rename(conflict_file, os.path.join(project_root, 'ad_diffi_backup.py'))

if project_src not in sys.path:
    sys.path.insert(0, project_src)
    print(f"Added to path: {project_src}")

# --- 4. File Fixes (core.py / benchmarks.py) ---
core_file_path = os.path.join(project_src, 'ad_diffi', 'core.py')
benchmarks_file_path = os.path.join(project_src, 'ad_diffi', 'benchmarks.py')

# FIX: Remove %%writefile from benchmarks.py
if os.path.exists(benchmarks_file_path):
    with open(benchmarks_file_path, 'r') as f:
        lines = f.readlines()
    if lines and '%%writefile' in lines[0]:
        print(f"Removing '%%writefile' from {benchmarks_file_path}")
        with open(benchmarks_file_path, 'w') as f:
            f.writelines(lines[1:])

# FIX: Ensure calculate_diffi_scores exists in core.py
if os.path.exists(core_file_path):
    with open(core_file_path, 'r') as f:
        core_content = f.read()
    if "def calculate_diffi_scores" not in core_content:
        print(f"Adding 'calculate_diffi_scores' to {core_file_path}")
        add_code = """
def calculate_diffi_scores(iforest, X, feature_names):
    n_samples, n_features = X.shape
    diffi_scores = np.zeros(n_features)
    initial_scores = iforest.decision_function(X)
    for i in range(n_features):
        X_perturbed = X.copy()
        np.random.shuffle(X_perturbed[:, i])
        perturbed_scores = iforest.decision_function(X_perturbed)
        diffi_scores[i] = np.sum(initial_scores - perturbed_scores)
    return diffi_scores
"""
        with open(core_file_path, 'a') as f:
            f.write(add_code)

# --- 5. Execute Import ---
try:
    for mod in list(sys.modules.keys()):
        if mod.startswith('ad_diffi'):
            del sys.modules[mod]

    import ad_diffi
    from ad_diffi.core import calculate_ad_diffi_zscore, diffi_ib_binary_rso, calculate_diffi_scores
    from ad_diffi.utils import download_adbench_dataset, get_feature_metadata
    from ad_diffi.benchmarks import run_diffi_comparison_benchmark

    print("\n[SUCCESS] All ad_diffi modules imported successfully from src/ad_diffi.")
    print(f"Imported from: {ad_diffi.__file__}")

except ImportError as e:
    print(f"\n[ERROR] Import failed: {e}")
    print("If still failing, try 'Runtime -> Restart session' and run this cell again.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 4.2 MB/s eta 0:00:00
Cloning AD-DIFFI...
Cloning into 'AD-DIFFI'...
remote: Enumerating objects: 314, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 314 (delta 91), reused 11 (delta 11), pack-reused 133 (from 1)
Receiving objects: 100% (314/314), 884.11 KiB | 3.32 MiB/s, done.
Resolving deltas: 100% (159/159), done.
Renaming /content/AD-DIFFI/ad_diffi.py to avoid import conflict.
Added to path: /content/AD-DIFFI/src
Adding 'calculate_diffi_scores' to /content/AD-DIFFI/src/ad_diffi/core.py

[SUCCESS] All ad_diffi modules imported successfully from src/ad_diffi.
Imported from: /content/AD-DIFFI/src/ad_diffi/__init__.py


## Cell 2: Theoretical Framework

## Chapter 5: Implementing AD-DIFFI — RSO and Standardization

### 5.1. Research Objective: Solving the Binary Bias
The primary objective of this analysis is to validate how the **AD-DIFFI** framework corrects the "Score Reversal" phenomenon. In the original DIFFI, highly efficient binary signals were paradoxically penalized because they isolated anomalies too quickly to accumulate "Inlier" scores in deeper nodes.

To resolve this structural bias, we implement and evaluate two primary mechanisms:

#### 5.1.1. Root-Split-Only (RSO) Constraint
Binary features are inherently limited in their splitting capacity (only 0 or 1). When a binary feature is used in deeper nodes of an Isolation Tree, it often represents random noise rather than a meaningful structural split. The **RSO Constraint** restricts the evaluation of binary features exclusively to the **root split (Depth 0)**. This prevents "noise accumulation" in the denominator and ensures that binary features are only rewarded for their primary discriminative power.

#### 5.1.2. Z-score Standardization
Binary and continuous features have fundamentally different probability distributions in an Isolation Forest. To compare them fairly, we establish a **Noise Baseline** by running the model on pure noise datasets.

We calculate the mean ($\mu_{noise}$) and standard deviation ($\sigma_{noise}$) of scores for each feature type separately. Every raw score is then converted into a **Z-score**:

$$Z = \frac{Score_{raw} - \mu_{noise}}{\sigma_{noise}}$$

This transformation creates a unified, zero-centered scale where any score above 0 indicates a signal stronger than random noise, regardless of the data type.

## Cell 3: Data Loading & Metadata Identification

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# =============================================================================
# 1. CONFIGURATION & SETUP
# =============================================================================
RANDOM_SEED = 42
DATASET_NAME = "hepatitis"
LABEL_COLUMN = "Outlier_label"

np.random.seed(RANDOM_SEED)

# =============================================================================
# 2. DATA ACQUISITION & PREPROCESSING (UCI Hepatitis)
# =============================================================================
def get_hepatitis_data():
    """
    Acquires and preprocesses the Hepatitis dataset from UCI.
    N=155, 19 Features (6 Continuous + 13 Binary).
    Outlier: Class 1 (Die), Normal: Class 2 (Live).
    """
    data_dir = Path('/tmp/ad_diffi_data')
    data_dir.mkdir(exist_ok=True, parents=True)
    csv_file = data_dir / 'hepatitis_processed.csv'

    if not csv_file.exists():
        print("Downloading Hepatitis dataset from UCI...")
        # UCI Repository direct link
        url = 'https://archive.ics.uci.edu/static/public/46/data.csv'

        # Define standard names based on UCI documentation
        cols = [
            'Class', 'AGE', 'SEX', 'STEROID', 'ANTIVIRALS', 'FATIGUE', 'MALAISE', 'ANOREXIA',
            'LIVER_BIG', 'LIVER_FIRM', 'SPLEEN_PALPABLE', 'SPIDERS', 'ASCITES', 'VARICES',
            'BILIRUBIN', 'ALK_PHOSPHATE', 'SGOT', 'ALBUMIN', 'PROTIME', 'HISTOLOGY'
        ]

        # Load data, treating '?' as NaN
        try:
            df_raw = pd.read_csv(url, na_values=['?'])
            # Match columns (Handling case where header might or might not exist)
            if len(df_raw.columns) == len(cols):
                df_raw.columns = cols
        except Exception as e:
            print(f"Error downloading: {e}. Attempting alternative loading...")
            # Fallback if URL structure changed
            df_raw = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/hepatitis/hepatitis.data',
                                 header=None, names=cols, na_values='?')

        # --- Preprocessing & Imputation ---
        continuous_features = ['AGE', 'BILIRUBIN', 'ALK_PHOSPHATE', 'SGOT', 'ALBUMIN', 'PROTIME']
        binary_features = [c for c in cols if c not in continuous_features + ['Class']]

        # 1. Handle Missing Values
        for col in continuous_features:
            df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
            df_raw[col] = df_raw[col].fillna(df_raw[col].median())

        for col in binary_features:
            df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
            df_raw[col] = df_raw[col].fillna(df_raw[col].mode()[0])

        # 2. Create Outlier Labels (Class 1=Die/Outlier, 2=Live/Normal)
        df_raw[LABEL_COLUMN] = np.where(df_raw['Class'] == 1, 'o', 'n')
        df_raw = df_raw.drop('Class', axis=1)

        df_raw.to_csv(csv_file, index=False)
        print(f"[SUCCESS] Hepatitis dataset processed: {len(df_raw)} samples.")

    return pd.read_csv(csv_file)

# Load the dataset
df = get_hepatitis_data()

# =============================================================================
# 3. METADATA IDENTIFICATION
# =============================================================================
# Extract feature names
feature_names = [col for col in df.columns if col != LABEL_COLUMN]

# Define types based on unique values
feature_types = []
for col in feature_names:
    # Most binary features in UCI are coded as 1/2 or 0/1
    if df[col].nunique() <= 2:
        feature_types.append('bin')
    else:
        feature_types.append('cont')

# Target vector y (1 for outliers 'o', 0 for normal 'n')
y = (df[LABEL_COLUMN] == 'o').astype(int).values

# =============================================================================
# 4. FINAL DISPLAY
# =============================================================================
n_cont = sum(1 for t in feature_types if t == 'cont')
n_bin = sum(1 for t in feature_types if t == 'bin')

print(f"--- DATASET STATISTICS: {DATASET_NAME.upper()} ---")
print(f"Total Samples:   {len(df)}")
print(f"Total Features:  {len(feature_names)}")
print(f" - Continuous Features: {n_cont}")
print(f" - Binary Features:     {n_bin}")
print(f"Anomaly Ratio:          {y.mean():.2%}")

print(f"\n[Feature Type Breakdown - First 10 Items]")
for i in range(min(10, len(feature_names))):
    print(f" - {feature_names[i]:<15}: {feature_types[i]}")

print("\n--- Feature Preview ---")
try:
    print(df[feature_names].head().to_markdown(index=False))
except:
    print(df[feature_names].head())

[SUCCESS] Hepatitis dataset processed: 155 samples.
--- DATASET STATISTICS: HEPATITIS ---
Total Samples:   155
Total Features:  19
 - Continuous Features: 6
 - Binary Features:     13
Anomaly Ratio:          20.65%

[Feature Type Breakdown - First 10 Items]
 - AGE            : cont
 - SEX            : bin
 - STEROID        : bin
 - ANTIVIRALS     : bin
 - FATIGUE        : bin
 - MALAISE        : bin
 - ANOREXIA       : bin
 - LIVER_BIG      : bin
 - LIVER_FIRM     : bin
 - SPLEEN_PALPABLE: bin

--- Feature Preview ---
|   AGE |   SEX |   STEROID |   ANTIVIRALS |   FATIGUE |   MALAISE |   ANOREXIA |   LIVER_BIG |   LIVER_FIRM |   SPLEEN_PALPABLE |   SPIDERS |   ASCITES |   VARICES |   BILIRUBIN |   ALK_PHOSPHATE |   SGOT |   ALBUMIN |   PROTIME |   HISTOLOGY |
|------:|------:|----------:|-------------:|----------:|----------:|-----------:|------------:|-------------:|------------------:|----------:|----------:|----------:|------------:|----------------:|-------:|----------:|----------:

## Cell 4: AD-DIFFI Implementation & Comparison Logic

In [ ]:
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from scipy.stats import spearmanr
import warnings

# =============================================================================
# 1. SETUP & OPTIMIZATION
# =============================================================================
if '/content/AD-DIFFI/src' not in sys.path:
    sys.path.insert(0, '/content/AD-DIFFI/src')

from ad_diffi.core import calculate_ad_diffi_zscore, calculate_diffi_scores, diffi_ib_binary_rso, get_noise_baselines
from ad_diffi.benchmarks import evaluate_feature_set_auc, get_top_k_features

def optimize_if_params_safe(df, feature_list, y):
    X_scaled = StandardScaler().fit_transform(df[feature_list].fillna(0).values)
    X_train, _, y_train, _ = train_test_split(X_scaled, y, test_size=0.3, stratify=y, random_state=42)

    n_train = X_train.shape[0]
    sample_candidates = [32, 64, n_train]

    param_grid = {
        'n_estimators': [100, 200],
        'max_samples': sample_candidates,
        'contamination': [0.15, 0.20, 0.25]
    }
    grid = GridSearchCV(IsolationForest(random_state=42), param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
    grid.fit(X_train, y_train)
    return grid.best_params_

best_params = optimize_if_params_safe(df, feature_names, y)
IF_PARAMS = {**best_params, "max_features": 1.0, "bootstrap": False}

# =============================================================================
# 2. CORE AD-DIFFI ANALYSIS
# =============================================================================
print(f"\n--- Running Analysis for: {DATASET_NAME.upper()} (N={len(df)}) ---")
X_scaled = StandardScaler().fit_transform(df[feature_names].values)
feature_types_dict = {i: feature_types[i] for i in range(len(feature_names))}

# Step A: Noise Baseline
c_mu, c_sigma, b_mu, b_sigma = get_noise_baselines(X_scaled.shape[1], feature_types_dict, IF_PARAMS, n_iter=20)
noise_baselines = {'cont': {'mean': c_mu, 'sd': c_sigma}, 'bin': {'mean': b_mu, 'sd': b_sigma}}

# Step B: Main Importance
orig_acc, rso_acc = [], []
for i in range(20):
    model = IsolationForest(**IF_PARAMS, random_state=i).fit(X_scaled)
    orig_acc.append(calculate_diffi_scores(model, X_scaled, feature_names))
    rso_acc.append(diffi_ib_binary_rso(model, X_scaled, feature_types_dict))

mean_orig = np.mean(orig_acc, axis=0)
mean_rso_z = calculate_ad_diffi_zscore(np.mean(rso_acc, axis=0), feature_types_dict, noise_baselines)

# Create DataFrame
compare_df = pd.DataFrame({
    "Feature": feature_names,
    "Type": feature_types,
    "Original_DIFFI": np.round(mean_orig, 4),
    "AD_DIFFI_RSO_Z": np.round(mean_rso_z, 4),
})
compare_df["Rank_Orig"] = compare_df["Original_DIFFI"].rank(ascending=False).astype(int)
compare_df["Rank_AD"] = compare_df["AD_DIFFI_RSO_Z"].rank(ascending=False).astype(int)
compare_df["Rank_Change"] = compare_df["Rank_Orig"] - compare_df["Rank_AD"]

print("\n### Feature Importance Comparison ###")
print(compare_df.sort_values("Rank_AD").to_markdown(index=False))

# =============================================================================
# 3. PERFORMANCE EVALUATION (IF & LR)
# =============================================================================
print("\n" + "=" * 80)
print("Performance Evaluation: IF AUC & LR AUC")
print("=" * 80)

TOP_K = 6
top_orig = get_top_k_features(compare_df, "Original_DIFFI", k=TOP_K)
top_ad = get_top_k_features(compare_df, "AD_DIFFI_RSO_Z", k=TOP_K)

def evaluate_lr_auc(df, features, y):
    X = StandardScaler().fit_transform(df[features].fillna(0))
    lr = LogisticRegression(solver='liblinear', random_state=42)
    from sklearn.metrics import roc_auc_score
    lr.fit(X, y)
    return roc_auc_score(y, lr.predict_proba(X)[:, 1])

results = []
for label, f_list in [("All Features", feature_names), (f"Original Top-{TOP_K}", top_orig), (f"AD-DIFFI Top-{TOP_K}", top_ad)]:
    if_auc = evaluate_feature_set_auc(df, f_list, target=y, model_type='IF', params=IF_PARAMS)
    lr_auc = evaluate_lr_auc(df, f_list, y)
    results.append([label, if_auc, lr_auc])

summary_df = pd.DataFrame(results, columns=["Method", "IF AUC", "LR AUC"])
print(summary_df.to_markdown(index=False))

# Final Stats
rho, _ = spearmanr(compare_df["Rank_Orig"], compare_df["Rank_AD"])
print(f"\nRank Stability (Spearman ρ): {rho:.4f}")
print(f"Mean |RankΔ|: {compare_df['Rank_Change'].abs().mean():.3f}")


--- Running Analysis for: HEPATITIS (N=155) ---

### Feature Importance Comparison ###
| Feature         | Type   |   Original_DIFFI |   AD_DIFFI_RSO_Z |   Rank_Orig |   Rank_AD |   Rank_Change |
|:----------------|:-------|-----------------:|-----------------:|------------:|----------:|--------------:|
| VARICES         | bin    |           0.2721 |           5.6183 |          11 |         1 |            10 |
| SEX             | bin    |           0.0452 |           4.2655 |          19 |         2 |            17 |
| ASCITES         | bin    |           0.2831 |           4.0425 |           9 |         3 |             6 |
| LIVER_BIG       | bin    |           0.2531 |           3.7089 |          13 |         4 |             9 |
| ANOREXIA        | bin    |           0.359  |           3.7008 |           6 |         5 |             1 |
| SPLEEN_PALPABLE | bin    |           0.3151 |           3.5972 |           7 |         6 |             1 |
| BILIRUBIN       | cont   |           0